In [1]:
import xarray as xr
import calendar

def download_nh_300mb_vwinds(start_year, start_month, end_year, end_month, output_file, hours_step=1):
    """
    Downloads Northern Hemisphere 300mb v-winds from ERA5 for a given time range.
    Note: Requires `gcsfs` and `zarr` to be installed (e.g., pip install gcsfs zarr)
    
    Args:
        start_year (int): Start year
        start_month (int): Start month
        end_year (int): End year
        end_month (int): End month
        output_file (str): Output NetCDF file name
        hours_step (int, optional): Temporal resolution in hours to subsample. 
                                    Default is 1 (native hourly resolution). 
                                    Use 3 for 3-hourly, 6 for 6-hourly, etc.
    """
    # ARCO-ERA5 pressure level dataset
    url = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
    
    print(f"Opening Zarr store at {url}...")
    # Using 'anon' token for public data access
    ds = xr.open_zarr(
        url,
        chunks=None,
        storage_options=dict(token='anon')
    )
    
    # Format the start and end dates
    start_date = f"{start_year}-{start_month:02d}-01"
    _, last_day = calendar.monthrange(end_year, end_month)
    end_date = f"{end_year}-{end_month:02d}-{last_day}"
    
    print(f"Selecting Northern Hemisphere 300 mb v-winds from {start_date} to {end_date}...")
    
    # Select northern hemisphere (latitude 90 to 0), 300 mb (level=300)
    # ERA5 latitudes are descending (90 to -90), so slice is from 90 to 0
    v_winds = ds['v_component_of_wind'].sel(
        level=300,
        latitude=slice(90, 0),
        time=slice(start_date, end_date)
    )
    
    # Subsample in time if requested
    if hours_step > 1:
        print(f"Subsampling to {hours_step}-hourly resolution...")
        # Since the native resolution is 1-hourly, taking every Nth index gives N-hourly data
        v_winds = v_winds.isel(time=slice(0, None, hours_step))
    
    print(f"Downloading and saving to {output_file}...")
    print(f"Shape of requested data: {v_winds.shape}")
    
    # Estimate size
    estimated_size_gb = v_winds.nbytes / (1024 ** 3)
    estimated_size_mb = v_winds.nbytes / (1024 ** 2)
    if estimated_size_gb >= 1:
        print(f"Estimated disk space required: {estimated_size_gb:.2f} GB")
    else:
        print(f"Estimated disk space required: {estimated_size_mb:.2f} MB")
    
    # Trigger download and save to NetCDF
    v_winds.to_netcdf(output_file)
    print(f"Done! Saved to {output_file}")

In [2]:
# Example usage:
download_nh_300mb_vwinds(
    start_year=2022, 
    start_month=11, 
    end_year=2023, 
    end_month=2,
    output_file="v_winds_300mb_nh_2022_2023.nc",
    hours_step=3 # Subsample to 3-hourly resolution
)

Opening Zarr store at gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3...


/Users/joymonteiro/miniconda3/envs/waper/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/joymonteiro/miniconda3/envs/waper/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


Selecting Northern Hemisphere 300 mb v-winds from 2022-11-01 to 2023-02-28...
Subsampling to 3-hourly resolution...
Shape of requested data: (960, 361, 1440)
Estimated disk space required: 1.86 GB
